In [ ]:
#| default_exp training.multigpu_training

In [ ]:
#| hide
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
import tensorflow as tf

In [ ]:
class BasnetLoss(tf.keras.losses.Loss):
    """BASNet hybrid loss."""

    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.smooth = 1.0e-9

        # Binary Cross Entropy loss.
        #self.cross_entropy_loss = focal_tversky_loss_r
        # Structural Similarity Index value.
        self.ssim_value = tf.image.ssim
        #  Jaccard / IoU loss.
        self.iou_value = self.calculate_iou

    def calculate_iou(
        self,
        y_true,
        y_pred,
    ):
        """Calculate intersection over union (IoU) between images."""
        intersection = K.sum(K.abs(y_true * y_pred), axis=[1, 2, 3])
        union = K.sum(y_true, [1, 2, 3]) + K.sum(y_pred, [1, 2, 3])
        union = union - intersection
        return K.mean(
            (intersection + self.smooth) / (union + self.smooth), axis=0
        )

    def call(self, y_true, y_pred):
        #cross_entropy_loss = focal_tversky_loss_r(y_true, y_pred)

        ssim_value = self.ssim_value(y_true, y_pred, max_val=1)
        ssim_loss = K.mean(1 - ssim_value + self.smooth, axis=0)

        iou_value = self.iou_value(y_true, y_pred)
        iou_loss = 1 - iou_value

        # Add all three losses.
        return ssim_loss + iou_loss

In [ ]:
loss_fn = BasnetLoss(reduction=tf.keras.losses.Reduction.NONE)

In [ ]:
#| export
import sys
from pathlib import Path
from platform import system
import mlflow
from datetime import datetime

In [ ]:
#| export
import tensorflow as tf
import tensorflow_addons as tfa
#import tensorflow_datasets as tfds
import tensorflow.keras.backend as K

ModuleNotFoundError: No module named 'keras.src.engine'

In [ ]:
#| export
CV_TOOLS = Path(r'/home/ai_sintercra/homes/hasan/projects/git_data/cv_tools')
CURRENT_REPO = Path(r'/home/ai_sintercra/homes/hasan/projects/git_data/new_test')
sys.path.append(str(CV_TOOLS))
sys.path.append(str(CURRENT_REPO))

In [ ]:
#| export
custom_lib_path = Path(r'/home/ai_warstein/homes/goni/custom_libs')
sys.path.append(str(custom_lib_path))

In [ ]:
#| export
from cv_tools.imports import *
from dotenv import load_dotenv
from argparse import ArgumentParser
from cv_tools.core import *

In [ ]:
#| export
load_dotenv(dotenv_path=f'/home/ai_sintercra/homes/hasan/projects/git_data/new_test/new_test/.env')

False

In [ ]:
#| export
CURRETNT_NB='/home/ai_sintercra/homes/hasan/projects/git_data/new_test/nbs'

In [ ]:
#| exporti
from new_test.preprocessing.create_tf_dataset import *

In [ ]:
config_path = Path(Path.cwd().parent, 'configs')
config_fn = 'config_office_local.yaml'
if system() == 'Windows':
    CONFIG_PATH=Path(config_path, config_fn)
config = get_config(CONFIG_PATH)

In [ ]:
trn_ds, val_ds = get_data(config)

 Number of mask files = 17894
 Number of image files = 17894
 Number of common files = 17894
Augmented image shape = (1152, 1632, 1)
Augmented image shape = (1152, 1632, 1)
 Number of mask files = 2687
 Number of image files = 2686
 Number of common files = 2686


NameError: name '_build_model' is not defined

In [ ]:
#| export
def tversky(y_true, y_pred, smooth=1):
    y_true_pos = K.flatten(y_true)
    y_pred_pos = K.flatten(y_pred)
    true_pos = K.sum(y_true_pos * y_pred_pos)
    false_neg = K.sum(y_true_pos * (1-y_pred_pos))
    false_pos = K.sum((1-y_true_pos)*y_pred_pos)
    alpha = 0.8
    return (true_pos + smooth)/(true_pos + alpha*false_neg + (1-alpha)*false_pos + smooth)

def tversky_loss(y_true, y_pred):
    return 1 - tversky(y_true,y_pred)

def focal_tversky_loss_r(y_true,y_pred):
    pt_1 = tversky(y_true, y_pred)
    gamma = 2
    return K.pow((1-pt_1), gamma)

In [ ]:
#| export
class BasnetLoss(tf.keras.losses.Loss):
    """BASNet hybrid loss."""

    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.smooth = 1.0e-9

        # Binary Cross Entropy loss.
        self.cross_entropy_loss = focal_tversky_loss_r
        # Structural Similarity Index value.
        self.ssim_value = tf.image.ssim
        #  Jaccard / IoU loss.
        self.iou_value = self.calculate_iou

    def calculate_iou(
        self,
        y_true,
        y_pred,
    ):
        """Calculate intersection over union (IoU) between images."""
        intersection = K.sum(K.abs(y_true * y_pred), axis=[1, 2, 3])
        union = K.sum(y_true, [1, 2, 3]) + K.sum(y_pred, [1, 2, 3])
        union = union - intersection
        return K.mean(
            (intersection + self.smooth) / (union + self.smooth), axis=0
        )

    def call(self, y_true, y_pred):
        #cross_entropy_loss = self.cross_entropy_loss(y_true, y_pred)
        cross_entropy_loss = focal_tversky_loss_r(y_true, y_pred)

        ssim_value = self.ssim_value(y_true, y_pred, max_val=1)
        ssim_loss = K.mean(1 - ssim_value + self.smooth, axis=0)

        iou_value = self.iou_value(y_true, y_pred)
        iou_loss = 1 - iou_value

        # Add all three losses.
        return cross_entropy_loss + ssim_loss + iou_loss

In [ ]:
#| export
def get_model(config: dict):
    """Build or load a segmentation model based on config.
    
    Args:
        config (dict): Configuration dictionary containing model specs
        
    Returns:
        tf.keras.Model: Compiled model ready for training
    """
    # Get model input parameters
    input_shape = tuple(config['model']['input_shape'])
    num_classes = config['model']['num_classes']
    
    if not config['model']['pretrained']:
        # Build new model based on architecture
        architecture = config['model']['architecture']
        model_map = {
            'UNet': encoder_decoder,
            'Unet256_Min350Pixel': encoder_decoder_256,
            'default': encode_128_decoder_128
        }
        
        model_fn = model_map.get(architecture, model_map['default'])
        if architecture == 'Unet256_Min350Pixel':
            tf.print('Using patch size 256')
            
        model = model_fn(input_size=input_shape, n_classes=num_classes)
        
    else:
        # Load pretrained model
        optimizer = tfa.optimizers.AdamW(
            learning_rate=1e-3,
            weight_decay=1e-5,
            clipnorm=1.0,
        )
        
        custom_objects = {
            'optimizer': optimizer,
            'BasnetLoss': BasnetLoss,
            'focal_tversky_loss_r': focal_tversky_loss_r
        }
        
        model = tf.keras.models.load_model(
            config['model']['pretrained_model_path'],
            custom_objects=custom_objects
        )
        
        tf.print('Loading pretrained model')
        
        # Freeze layers if specified
        trainable_layers = config['model']['num_trainable_layers']
        if trainable_layers > 0:
            for layer in model.layers[:trainable_layers]:
                layer.trainable = False
            tf.print(f'Froze {trainable_layers} layers')
            
    # Enable mixed precision for faster training
    if tf.config.list_physical_devices('GPU'):
        tf.keras.mixed_precision.set_global_policy('mixed_float16')
            
    return model

In [ ]:
#| export
def model_training(
    config:dict,
    trn_ds,
    val_ds,
    loss_fn,
    optimizer,
    verbose:int=1,
    experiment_name:str='easy_pin_detectionV02',
    ):

    # Set up multi-GPU strategy if available
    gpus = tf.config.list_physical_devices('GPU')
    if len(gpus) > 1:
        strategy = tf.distribute.MirroredStrategy()
        print(f'Training using {len(gpus)} GPUs in MirroredStrategy')
    else:
        strategy = tf.distribute.get_strategy() # Default strategy
        print('Training on single device')

    with strategy.scope():
        # Enable mixed precision for faster training
        tf.keras.mixed_precision.set_global_policy('mixed_float16')

        MODEL_EXPERIMENT_NAME=experiment_name
        mlflow.set_experiment(MODEL_EXPERIMENT_NAME)

        trn_images = len(Path(config['dataset']['trn_im_path']).ls(file_exts=['.png', '.jpg', '.jpeg', '.tif', '.tiff']))
        val_images = len(Path(config['dataset']['val_im_path']).ls(file_exts=['.png', '.jpg', '.jpeg', '.tif', '.tiff']))

        # Calculate steps with larger batch size for multi-GPU
        global_batch_size = config['training']['batch_size'] * (len(gpus) if len(gpus) > 1 else 1)
        config['training']['steps_per_epoch'] = trn_images // global_batch_size
        config['training']['validation_steps'] = val_images // global_batch_size

        if trn_images % global_batch_size > 0:
            config['training']['steps_per_epoch'] += 1
        if val_images % global_batch_size > 0:
            config['training']['validation_steps'] += 1

        # Set up model checkpointing
        current_datetime = datetime.now()
        time = current_datetime.strftime("%H_%M_%S")
        Path(config['model']['save_path']).mkdir(parents=True, exist_ok=True)
        model_save_path=config['model']['save_path']

        model_checkpoint = tf.keras.callbacks.ModelCheckpoint(
            monitor='val_foreground',
            filepath=f'{model_save_path}/time_{time}_val_frGrnd'+'{val_foreground:.4f}_epoch_{epoch}.h5',
            save_best_only=True,
            save_freq='epoch',
            mode='max'
        )

        # Add performance optimization callbacks
        reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
            monitor='val_loss', 
            factor=0.5,
            patience=5,
            min_lr=1e-6,
        verbose=1
        )

        early_stopping = tf.keras.callbacks.EarlyStopping(
            monitor='val_foreground',
            patience=15,
            restore_best_weights=True
        )

        with mlflow.start_run():
            # Optimize learning rate schedule
            model = get_model(config)
            learning_rate_schedule = tf.keras.optimizers.schedules.CosineDecayRestarts(
                initial_learning_rate=config['training']['initial_learning_rate'],
                first_decay_steps=config['training']['first_decay_steps'],
                t_mul=config['training']['t_mul'],
                m_mul=config['training']['m_mul'],
                alpha=config['training']['alpha']
            )

            # Enable XLA compilation for faster training
            #tf.config.optimizer.set_jit(True)

            model.compile(
                loss=loss_fn,
                optimizer=optimizer,
                metrics=[
                    tf.keras.metrics.BinaryIoU(
                        name='foreground',
                        target_class_ids=[1],
                        threshold=0.5
                    )
                ]
            )

            # Enable caching and prefetching for faster data loading
            AUTOTUNE = tf.data.AUTOTUNE
            trn_ds = trn_ds.cache().prefetch(AUTOTUNE)
            val_ds = val_ds.cache().prefetch(AUTOTUNE)

            callbacks = [
                model_checkpoint,
                reduce_lr,
                early_stopping,
                tf.keras.callbacks.TerminateOnNaN()
            ]

            history = model.fit(
                trn_ds,
                validation_data=val_ds,
                steps_per_epoch=config['training']['steps_per_epoch'],
                validation_steps=config['training']['validation_steps'],
                epochs=config['training']['epochs'],
                callbacks=callbacks
            )

            # Log metrics
            for epoch in range(len(history.history['loss'])):
                mlflow.log_metric('train_loss', history.history['loss'][epoch], step=epoch)
                mlflow.log_metric('val_loss', history.history['val_loss'][epoch], step=epoch)
                mlflow.log_metric('foreground', history.history['foreground'][epoch], step=epoch)
                mlflow.log_metric('val_foreground', history.history['val_foreground'][epoch], step=epoch)

In [ ]:
#| export
def parse_args_():
    parser = ArgumentParser()
    parser.add_argument(
        '--config_path', type=str, required=True, help='Path to the config file'
    )
    parser.add_argument(
        '--experiment_name', type=str, required=True, help='Name of the experiment'
    )
    return parser.parse_args()

In [ ]:
#| export
def main_():
    args = parse_args_()
    config_path = Path(args.config_path)
    experiment_name = args.experiment_name
    config = get_config(config_path)
    trn_ds, val_ds = get_data(config)
    loss_fn = BasnetLoss()
    optimizer = tfa.optimizers.AdamW(
        learning_rate=1e-3, 
        weight_decay=1e-5, 
        clipnorm=1.0) 
    model = get_model(config)
    model_training(
        config, 
        trn_ds, 
        val_ds, 
        loss_fn, 
        optimizer, 
        experiment_name
    )

In [ ]:
#| export
if __name__ == '__main__':
    main_()

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export('38_training.multigpu_training.ipynb')

ValueError: '\\\\vihsdv140.infineon.com\\ai_sintercra\\homes\\hasan\\projects\\git_data\\new_test\\new_test\\nbs\\13_noise_lighting.ipynb' is not in the subpath of '\\\\vihsdv140.infineon.com\\ai_sintercra\\homes\\hasan\\projects\\git_data\\new_test\\nbs' OR one path is relative and the other is absolute.